In [5]:
# 1. Spark imports and launches
from pyspark.sql import SparkSession
import numpy as np

spark = SparkSession.builder.appName("LinearRegressionGradientDescent").getOrCreate()
sc = spark.sparkContext

In [6]:
# 2. Data generation: X ∈ ℝ^(100000×3), y = Xw + noise
n_samples = 100000
n_features = 3
true_weights = np.array([1.5, 0.3, -0.7])

X = np.random.randn(n_samples, n_features)
noise = 0.1 * np.random.randn(n_samples)
y = X @ true_weights + noise

data = list(zip(X.tolist(), y.tolist()))
rdd = sc.parallelize(data)

In [7]:
# 3. Gradient descent
def gradient_descent(rdd, learning_rate=0.1, n_iters=50):
    weights = np.zeros(n_features)
    
    for i in range(n_iters):
        gradient = rdd.map(lambda row: np.multiply(row[0], (np.dot(row[0], weights) - row[1]))) \
                      .reduce(lambda a, b: a + b) / n_samples
        weights -= learning_rate * gradient
        if i % 10 == 0:
            loss = rdd.map(lambda row: (np.dot(row[0], weights) - row[1])**2).mean()
            print(f"Iteration {i}: Loss = {loss}")
    
    return weights

In [8]:
# 4. Starting training
estimated_weights = gradient_descent(rdd)
print("Estimated weights:", estimated_weights)
print("True weights:", true_weights)

Iteration 0: Loss = 2.3016799647585455
Iteration 10: Loss = 0.28862195618527786
Iteration 20: Loss = 0.04384675717155685
Iteration 30: Loss = 0.01407657335930752
Iteration 40: Loss = 0.01045497811132517
Estimated weights: [ 1.49228029  0.2983942  -0.69599025]
True weights: [ 1.5  0.3 -0.7]
